#Task 3: User-based collaborative filtering

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

r_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings = pd.read_csv('u.data', sep='\t', names=r_cols, encoding='latin-1')

m_cols = ['movie_id', 'movie_title']
movies = pd.read_csv('u.item', sep='|', names=m_cols, usecols=[0, 1], encoding='latin-1')

print("Data loaded successfully.")

Data loaded successfully.


In [ ]:
train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)

train_matrix = train_data.pivot(index='user_id', columns='movie_id', values='rating').fillna(0)
user_sim = cosine_similarity(train_matrix)
user_sim_df = pd.DataFrame(user_sim, index=train_matrix.index, columns=train_matrix.index)

print(f"Training Matrix Shape: {train_matrix.shape}")

Training Matrix Shape: (943, 1653)


In [ ]:
def get_recommendations(user_id, n=10):
    if user_id not in train_matrix.index:
        return []

    # Find 5 most similar users
    sim_users = user_sim_df[user_id].sort_values(ascending=False).index[1:6]

    # Movies the user has already seen in training
    seen_movies = set(train_data[train_data['user_id'] == user_id]['movie_id'])

    recommendations = []
    for neighbor in sim_users:
        # Get neighbor's top rated movies (rating >= 4)
        neighbor_likes = train_data[(train_data['user_id'] == neighbor) & (train_data['rating'] >= 4)]['movie_id']
        new_items = set(neighbor_likes) - seen_movies
        recommendations.extend(list(new_items))
        if len(set(recommendations)) >= n:
            break

    # Return unique Top N IDs
    return list(set(recommendations))[:n]

In [ ]:
def calculate_metrics(n_users=100, k=10):
    all_precision = []
    all_recall = []

    # Get users who exist in both sets
    common_users = set(train_data['user_id']).intersection(set(test_data['user_id']))

    for user_id in list(common_users)[:n_users]:
        #Get Ground Truth: Movies user liked in test set (Rating >= 4)
        actual_relevant = set(test_data[(test_data['user_id'] == user_id) & (test_data['rating'] >= 4)]['movie_id'])

        if len(actual_relevant) == 0:
            continue

        #Get Recommended items
        recommended = set(get_recommendations(user_id, n=k))

        #Calculate Hits (Intersection)
        hits = len(recommended.intersection(actual_relevant))

        precision = hits / k

        recall = hits / len(actual_relevant)

        all_precision.append(precision)
        all_recall.append(recall)

    avg_p = np.mean(all_precision)
    avg_r = np.mean(all_recall)
    f1 = (2 * avg_p * avg_r) / (avg_p + avg_r) if (avg_p + avg_r) > 0 else 0

    return avg_p, avg_r, f1

# Execute evaluation
prec, rec, f1 = calculate_metrics(n_users=200, k=10)

print("--- Task 3: Performance Metrics ---")
print(f"Average Precision @ 10: {prec:.4f}")
print(f"Average Recall @ 10:    {rec:.4f}")
print(f"F1-Score:               {f1:.4f}")

--- Task 3: Performance Metrics ---
Average Precision @ 10: 0.0933
Average Recall @ 10:    0.1257
F1-Score:               0.1071


#Task 4: Item-based collaborative filtering

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load Data
r_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings = pd.read_csv('u.data', sep='\t', names=r_cols, encoding='latin-1')

m_cols = ['movie_id', 'movie_title']
movies = pd.read_csv('u.item', sep='|', names=m_cols, usecols=[0, 1], encoding='latin-1')

# 2. Train/Test Split (80/20)
train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)

# 3. Create Item-User Matrix (Rows = Movies, Columns = Users)
item_user_matrix = train_data.pivot(index='movie_id', columns='user_id', values='rating').fillna(0)

# 4. Compute Item Similarity Matrix
# We calculate Cosine Similarity across the rows (movies)

item_sim = cosine_similarity(item_user_matrix)
item_sim_df = pd.DataFrame(item_sim, index=item_user_matrix.index, columns=item_user_matrix.index)

print(f"Item-User Matrix Shape: {item_user_matrix.shape}")
print("Item Similarity Matrix calculated successfully.")

Item-User Matrix Shape: (1653, 943)
Item Similarity Matrix calculated successfully.


In [ ]:
def get_item_based_recommendations(user_id, n=10):
    if user_id not in train_data['user_id'].unique():
        return []

    # 1. Get movies the user liked in the training set (rating >= 4)
    user_likes = train_data[(train_data['user_id'] == user_id) & (train_data['rating'] >= 4)]['movie_id'].tolist()

    # Movies the user has already seen (to exclude them later)
    user_seen = set(train_data[train_data['user_id'] == user_id]['movie_id'])

    if not user_likes:
        return []

    # 2. Find similar movies to the ones the user liked
    similar_scores = pd.Series(dtype=float)

    for movie_id in user_likes:
        if movie_id in item_sim_df.index:
            # Get similarities for this movie, drop the movie itself, and add to our tracking series
            sims = item_sim_df[movie_id].drop(movie_id)
            # Add scores (if a movie is similar to multiple liked movies, its score goes up)
            similar_scores = similar_scores.add(sims, fill_value=0)

    # 3. Sort by aggregated similarity score
    similar_scores = similar_scores.sort_values(ascending=False)

    # 4. Filter out movies the user has already seen
    recommendations = [m_id for m_id in similar_scores.index if m_id not in user_seen]

    # Return Top N
    return recommendations[:n]

test_user = 1
print(f"Item-Based Recommendations for User {test_user} (Movie IDs):")
print(get_item_based_recommendations(test_user, n=5))

Item-Based Recommendations for User 1 (Movie IDs):
[204, 186, 238, 423, 100]


In [ ]:
def evaluate_item_based(n_users=100, k=10):
    all_precision = []
    all_recall = []

    common_users = set(train_data['user_id']).intersection(set(test_data['user_id']))

    for user_id in list(common_users)[:n_users]:
        # Ground Truth from Test Set
        actual_relevant = set(test_data[(test_data['user_id'] == user_id) & (test_data['rating'] >= 4)]['movie_id'])

        if len(actual_relevant) == 0:
            continue

        # Predictions
        recommended = set(get_item_based_recommendations(user_id, n=k))

        if len(recommended) == 0:
            continue

        # Intersection
        hits = len(recommended.intersection(actual_relevant))

        # Metrics
        precision = hits / k
        recall = hits / len(actual_relevant)

        all_precision.append(precision)
        all_recall.append(recall)

    avg_p = np.mean(all_precision)
    avg_r = np.mean(all_recall)
    f1 = (2 * avg_p * avg_r) / (avg_p + avg_r) if (avg_p + avg_r) > 0 else 0

    return avg_p, avg_r, f1

# Execute evaluation
prec_item, rec_item, f1_item = evaluate_item_based(n_users=200, k=10)

print("--- Task 4: Item-Based CF Metrics ---")
print(f"Average Precision @ 10: {prec_item:.4f}")
print(f"Average Recall @ 10:    {rec_item:.4f}")
print(f"F1-Score:               {f1_item:.4f}")

--- Task 4: Item-Based CF Metrics ---
Average Precision @ 10: 0.2190
Average Recall @ 10:    0.2483
F1-Score:               0.2327
